# Notebook 01: Baseline Exploration

Explore native baseline performance across all platforms and kernels.
Verify roofline utilization ≥ 80% before proceeding to abstraction experiments.

**Prerequisite:** `data/performance.csv` populated with native rows.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PEAK_BW = {'nvidia_a100': 2039, 'amd_mi250x': 3277, 'intel_pvc': 3276}

df = pd.read_csv('../data/performance.csv', parse_dates=['timestamp'])
native = df[df['abstraction'] == 'native'].copy()
print(f"Native rows: {len(native)}")
native.groupby(['kernel', 'platform', 'problem_size'])['throughput'].median().unstack()

In [ ]:
# Roofline utilization check
stream_large = native[(native['kernel'] == 'stream') & (native['problem_size'] == 'large')]
for platform, grp in stream_large.groupby('platform'):
    median_bw = grp['throughput'].median()
    util = median_bw / PEAK_BW.get(platform, float('nan'))
    status = 'PASS' if util >= 0.80 else 'FAIL'
    print(f"{platform}: {median_bw:.0f} GB/s  ({util:.1%} of peak)  [{status}]")